# 00 — Warm-up: your first model calls

Before agents, harnesses, or tools: just **you and the model**. Run this
top to bottom once — if the last cell works, your setup is ready for the week.

*GIU AI Connects · Amr Saleh · iHQ Tech*

## 0 · Setup

One `uv` environment for the whole week (see `README.md`): `uv sync`, then run
Jupyter with `uv run jupyter lab`.

Put your API key in a `.env` file next to this notebook:

```
ANTHROPIC_API_KEY=sk-ant-...
```

> Using another provider? Swap the model string below — e.g. `"openai:gpt-4.1"`
> with `OPENAI_API_KEY` in `.env`. Everything else stays identical: that is the
> point of the harness.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain_anthropic import ChatAnthropic

llm = ChatAnthropic(model="claude-haiku-4-5")

reply = llm.invoke("Say hello to the GIU AI Connects workshop in one sentence.")
reply.content

## What came back is not a string

`invoke` returns an **AIMessage** — the reply plus everything around it:
the text, the token usage, the stop reason. Poke at it:

In [ ]:
print(type(reply).__name__)
print(reply.content)
print(reply.usage_metadata)     # you pay per token — get used to seeing this
print(reply.response_metadata.get("stop_reason"))

## Simple things it does well

Text in, text out — that is the whole interface today.

In [ ]:
print(llm.invoke("Translate to German, reply with the translation only: "
                 "'Where is the lecture hall?'").content)

In [ ]:
notes = """Meeting notes: The workshop runs Tue-Fri. Tuesday covers agents from
scratch. Wednesday is production topics. Thursday shows real systems and evals.
Friday is office hours. Students should bring laptops and an API key."""

print(llm.invoke(f"Summarize in exactly one sentence: {notes}").content)

In [ ]:
code_snippet = """
def f(xs):
    return {x: xs.count(x) for x in set(xs)}
"""
print(llm.invoke(f"Explain what this does, in two sentences, to a first-year student:\n{code_snippet}").content)

## Two knobs worth knowing

**temperature** — how adventurous the sampling is. Run the next cell twice:
at `0` the answers repeat; at `1` they wander.

In [ ]:
strict = ChatAnthropic(model="claude-haiku-4-5", temperature=0)
loose  = ChatAnthropic(model="claude-haiku-4-5", temperature=1)

print("t=0:", strict.invoke("Name a city for a robotics conference. Reply with the city only.").content)
print("t=1:", loose.invoke("Name a city for a robotics conference. Reply with the city only.").content)

In [ ]:
# max_tokens — a hard cap on the reply length (it CUTS, not summarizes)
capped = ChatAnthropic(model="claude-haiku-4-5", max_tokens=30)
r = capped.invoke("Explain what a neural network is.")
print(r.content)
print("stop_reason:", r.response_metadata.get("stop_reason"))   # 'max_tokens' = it was cut

## Streaming — why chat apps feel alive

The model produces tokens one by one; `stream` hands them to you as they come.

In [ ]:
for chunk in llm.stream("Count from 1 to 10, one number per line."):
    print(chunk.content, end="", flush=True)

## A taste of tomorrow

Everything this week is built from **lists of messages**. Here is the shape —
day 2 starts exactly here:

In [ ]:
from langchain.messages import SystemMessage, HumanMessage

reply = llm.invoke([
    SystemMessage("You answer in exactly three words."),
    HumanMessage("What is an AI agent?"),
])
reply.content

> One more thing: from day 2 on we write `init_chat_model("anthropic:claude-haiku-4-5")`
> instead of `ChatAnthropic(...)` — same object underneath, but the string form
> lets anyone in the room swap providers without changing any other code.

**Your turn** — three quick ones:
1. Make it reply ONLY in Arabic, using a `SystemMessage`.
2. Cap a poem at 15 tokens and print the `stop_reason`.
3. Ask the same creative question 3 times at `temperature=1` — how different
   are the answers?

In [ ]:
# your three experiments here


---
If everything above ran: **you are ready for day 2.** Tomorrow this model goes
inside `create_agent` — and gets hands.